# 3-2회차 | 분류 모델 평가지표

**핵심 질문**: 정확도 90%를 믿어도 되나? — digits의 **7 vs others**로 확인

**오늘의 목표**
1. **정확도(Accuracy)의 함정**: 왜 때로는 위험한가 (불균형 데이터에서 착시)
2. **혼동행렬(Confusion Matrix)**: 무엇을 많이 틀렸는지 보기 (TP/FP/FN/TN)
3. **정밀도(Precision), 재현율(Recall), F1**: 의미와 해석
4. **임곗값(Threshold)**: 바꾸면 지표가 어떻게 달라지는지
5. **ROC 곡선과 AUC, PR 곡선(AP)**: 전체 임곗값에서의 분류력 요약

---
## 3-2에서 평가지표를 다루는 이유

지난 회차에 나눈 **train / validation / test의 역할**을 오늘 threshold 선택에도 그대로 적용함.

3-1에서 남긴 질문도 이어짐.

> **모델을 믿기 전에, 우리가 만든 실험을 먼저 의심해야 함.**

이번에는 한 번 나온 Accuracy를 너무 빨리 믿지 않고,
**무엇을 맞추고 무엇을 틀렸는지** 여러 Evidence로 확인할 것.

### 왜 정확도만 보면 안 되는가?

| 상황 | 정확도 | 실제 성능 |
|------|--------|----------|
| 암 환자 1%, 건강한 사람 99% | "모두 건강"이라고 예측 → 99% | 암 환자를 **전혀 못 찾음** |
| 스팸 10%, 정상 90% | "모두 정상"이라고 예측 → 90% | 스팸을 **전혀 못 걸러냄** |

> Accuracy는 출발점일 수 있지만, **한 숫자만으로 모델을 믿지는 않음.**

### 오늘 쓸 지표 요약

| 지표 | 의미 | 언제 중요한가 |
|------|------|-------------|
| **정확도(Accuracy)** | 전체 중 맞춘 비율 | 클래스가 균형일 때 |
| **정밀도(Precision)** | 양성 예측 중 실제 양성 | 오탐을 줄여야 할 때 (스팸) |
| **재현율(Recall)** | 실제 양성 중 찾은 비율 | 누락을 줄여야 할 때 (암) |
| **F1** | Precision과 Recall의 균형 | 둘 다 중요할 때 |
| **ROC-AUC** | 전체 임곗값에서 분류력 | 균형 데이터 성능 요약 |
| **PR-AP** | 양성 기준 분류력 | 불균형 데이터 성능 요약 |

In [1]:
import sklearn, numpy as np, pandas as pd
import plotly.express as px
import plotly.graph_objects as go
print("sklearn:", sklearn.__version__)
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)

sklearn: 1.6.1
numpy  : 2.4.2
pandas : 2.3.2


---
## Part 1. Accuracy(정확도)의 함정

### Titanic 데이터로 Dummy 분류기 만들기

**Titanic 데이터셋 소개**
- 실제 타이타닉호 생존자 데이터를 정리한 유명한 공개 데이터셋
- 목표: 승객의 나이, 성별, 선실 등급 등을 보고 **생존 여부(0=사망, 1=생존)**를 예측
- 주요 컬럼:
  - `Pclass`: 선실 등급 (1=일등석, 2=이등석, 3=삼등석)
  - `Sex`: 성별
  - `Age`: 나이
  - `Fare`: 운임 요금
- **Target(정답)**: `Survived` (1=생존, 0=사망)

> **3-1 회수 — 결측치를 채우는 값은 어디서 계산함?**
>
> 아래 셀에서 `Age` 결측을 평균으로 채움. 그 평균을 **train에서만** 계산함.
> test의 평균을 쓰면 test 정보가 전처리에 새어 들어감.
>
> 오늘 Part 1의 Dummy는 `Sex`만 보기 때문에 이 선택이 숫자를 바꾸지는 않음.
> 그래도 **fit 괄호 안을 보는 습관**은 여기서부터 지킴.

In [2]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

CSV = "train.csv"
if not Path(CSV).exists():
    raise FileNotFoundError(
        f"'{CSV}' 를 이 노트북과 같은 폴더에 두고 다시 실행하셈. (Kaggle Titanic train.csv)"
    )

def fillna(df, age_fill, fare_fill=0):
    df = df.copy()
    df['Age'] = df['Age'].fillna(age_fill)
    df['Cabin'] = df['Cabin'].fillna('N')
    df['Embarked'] = df['Embarked'].fillna('N')
    df['Fare'] = df['Fare'].fillna(fare_fill)
    return df

def drop_features(df):
    return df.drop(['PassengerId','Name','Ticket'], axis=1)

def format_features(df):
    df = df.copy()
    df['Cabin'] = df['Cabin'].astype(str).str[:1]
    return df

def transform_features(df, age_fill):
    return format_features(drop_features(fillna(df, age_fill)))

titanic_df = pd.read_csv(CSV)

train_df, test_df = train_test_split(
    titanic_df, test_size=0.2, stratify=titanic_df['Survived'], random_state=0
)

y_train_titanic = train_df['Survived']
y_test_titanic  = test_df['Survived']

# 결측치를 채우는 값은 train에서만 계산함 (test 통계를 쓰지 않음)
age_fill_value = train_df['Age'].mean()
print(f"Age 결측 대체값(train 평균만 사용): {age_fill_value:.4f}")

X_train_titanic = transform_features(train_df.drop('Survived', axis=1), age_fill_value)
X_test_titanic  = transform_features(test_df.drop('Survived', axis=1), age_fill_value)


Age 결측 대체값(train 평균만 사용): 29.4698


### Dummy Classifier (항상 같은 규칙만 적용)
- 성별이 male 이면 무조건 0(사망), 아니면 1(생존)으로 예측.
- **주의**: 이건 "학습"이 아니라 그냥 규칙임!

In [3]:
from sklearn.base import BaseEstimator
from sklearn.metrics import accuracy_score

class MyDummyClassifier(BaseEstimator):
    def fit(self, X, y=None): return self
    def predict(self, X):
        return np.where(X['Sex'].values=="male", 0, 1).astype(int)

pred = MyDummyClassifier().fit(X_train_titanic, y_train_titanic).predict(X_test_titanic)
print('Dummy Classifier 정확도:', accuracy_score(y_test_titanic, pred))

Dummy Classifier 정확도: 0.776536312849162


> **77%나 나왔다고?**  
> 하지만 이건 그냥 "남자면 사망"이라는 규칙일 뿐!  
> 정확도만 보면 **착시**에 빠질 수 있음!

### 불균형 데이터란?

- 한 클래스가 **극소수**인 상황 (예: 사기, 암 환자, 스팸)
- 이때 Accuracy는 쉽게 높아지므로 **지표를 다양하게** 봐야 함

| 상황 | 양성 비율 | "모두 음성" 예측 시 정확도 |
|------|----------|------------------------|
| 암 진단 | 1% | 99% |
| 금융 사기 | 0.1% | 99.9% |
| 스팸 메일 | 10% | 90% |

> **정확도가 높아도 양성을 전혀 못 찾으면 의미가 없음!**

---
## Part 2. 불균형 데이터 예 — digits에서 '7' vs 나머지

- **Positive = 숫자 7**, Negative = 나머지 숫자
- 목표: 불균형 데이터에서 "전부 0만 예측해도 정확도가 높게 보이는" 착시를 직접 확인

In [4]:
from sklearn.datasets import load_digits

digits = load_digits()
X_all_7 = digits.data
y_all_7 = (digits.target == 7).astype(int)  # 7이면 1, 아니면 0

X_train_7, X_test_7, y_train_7, y_test_7 = train_test_split(
    X_all_7, y_all_7, test_size=0.2, stratify=y_all_7, random_state=11
)

print(f"테스트셋 클래스 비율:\n{pd.Series(y_test_7).value_counts(normalize=True)}")

테스트셋 클래스 비율:
0    0.9
1    0.1
Name: proportion, dtype: float64


In [5]:
import plotly.express as px

dist_df_7 = pd.DataFrame({"label": y_test_7})
fig_dist_7 = px.histogram(
    dist_df_7,
    x="label",
    color="label",
    text_auto=True,
    title="digits (test) — Class Distribution (7 vs others)",
    category_orders={"label": [0, 1]},
    template="plotly_dark",
)
fig_dist_7.update_xaxes(title="Label (0=others, 1=seven)")
fig_dist_7.update_yaxes(title="Count")
fig_dist_7.show()

### Dummy(All-0) 결과: Accuracy는 높지만, 양성을 전혀 못 맞춤

In [6]:
from sklearn.metrics import precision_score, recall_score

class FakeClassifier(BaseEstimator):
    def fit(self, X, y=None):
        return self

    def predict(self, X):
        return np.zeros(len(X), dtype=int)

fakepred_7 = FakeClassifier().fit(X_train_7, y_train_7).predict(X_test_7)

print(f"Accuracy: {accuracy_score(y_test_7, fakepred_7):.3f}")
print("Precision:", precision_score(y_test_7, fakepred_7, zero_division=0))
print("Recall   :", recall_score(y_test_7, fakepred_7, zero_division=0))

Accuracy: 0.900
Precision: 0.0
Recall   : 0.0


> **정확도 90%인데 Precision=0, Recall=0?**  
> 이게 바로 정확도의 **함정**임!  
> 양성(숫자 7)을 **하나도 못 찾았는데** 정확도는 높게 나옴

---
## 잠깐! 지표를 보기 전에 — "Positive"를 먼저 정하자

분류 문제에서 평가지표를 보려면, **내가 관심 있는 대상**을 먼저 정해야 함!

### 3단계로 이해하기

| 순서 | 내용 |
|------|------|
| 1 | **내가 알고 싶은(찾고 싶은) 대상**이 있다 |
| 2 | 그걸 **Positive**라고 부른다 |
| 3 | 지금부터 보는 지표들은 전부 **그 Positive를 기준으로 한 성적표**다 |

### 예시

| 문제 | Positive (관심 대상) | Negative (비교 대상) |
|------|---------------------|---------------------|
| 암 진단 | 암 환자 | 건강한 사람 |
| 스팸 필터 | 스팸 메일 | 정상 메일 |
| 사기 탐지 | 사기 거래 | 정상 거래 |
| digits 예제 | 숫자 7 | 나머지 숫자 |

> **실무 팁**: 모델을 만들기 전에 **"Positive class = ?"** 을 반드시 적고 시작하셈

### 그래서 지표는 어떻게 나뉘나?

- **Accuracy** = **전체 성적표** (Positive든 Negative든 전부 포함)
- **Recall, Precision, F1, ROC, PR** = **Positive(주인공) 중심 성적표**

| 지표 | Positive 관점에서 묻는 질문 |
|------|--------------------------|
| **Recall** | 놓치지 않았나? |
| **Precision** | 헛발질 안 했나? |
| **F1** | 둘의 균형은? |
| **ROC** | 기준값을 바꿔도 Positive랑 Negative를 잘 구분하나? |
| **PR** | Positive가 드문 상황에서도 의미 있게 찾나? |

> **한 문장 요약**: Positive는 관심 대상, Negative는 비교 대상임

![Accuracy는 전체 평균, 나머지는 주인공의 성적표](스크린샷%202026-02-09%20오후%202.32.44.png)

---
## Part 3. 혼동행렬 (Confusion Matrix)

### 혼동행렬 읽는 법

```
              예측=0    예측=1
실제=0         TN        FP
실제=1         FN        TP
```

| 약어 | 의미 | 설명 |
|------|------|------|
| **TN** | True Negative | 음성인데 음성으로 맞춤 |
| **FP** | False Positive | 음성인데 양성으로 틀림 (**오탐**) |
| **FN** | False Negative | 양성인데 음성으로 틀림 (**누락**) |
| **TP** | True Positive | 양성인데 양성으로 맞춤 |

> Dummy(all-zero)는 FN이 가득 → 양성을 전혀 못 잡는다(Recall≈0)

In [7]:
from sklearn.metrics import confusion_matrix, classification_report
import plotly.figure_factory as ff
import numpy as np

cm_7 = confusion_matrix(y_test_7, fakepred_7, labels=[0, 1])

ann_7 = np.array([
    [f"TN={cm_7[0,0]}", f"FP={cm_7[0,1]}"],
    [f"FN={cm_7[1,0]}", f"TP={cm_7[1,1]}"],
])

fig_cm_7 = ff.create_annotated_heatmap(
    z=cm_7,
    x=["예측=0", "예측=1"],
    y=["실제=0", "실제=1"],
    annotation_text=ann_7,
    colorscale="Blues",
    showscale=True,
    font_colors=["black", "white"],
)
fig_cm_7.update_layout(
    title="Confusion Matrix (Dummy: all-zero, 7 vs others)",
    xaxis=dict(title="Predicted"),
    yaxis=dict(title="Actual"),
    template="plotly_dark",
)
fig_cm_7.show()

print("TN, FP, FN, TP:", *cm_7.ravel())
print(classification_report(y_test_7, fakepred_7, digits=3, zero_division=0))

expected_dummy_cm_parity_7 = (324, 0, 36, 0)
actual_dummy_cm_parity_7 = tuple(cm_7.ravel())
assert actual_dummy_cm_parity_7 == expected_dummy_cm_parity_7, actual_dummy_cm_parity_7
print("dummy confusion-matrix script parity: PASS")

TN, FP, FN, TP: 324 0 36 0
              precision    recall  f1-score   support

           0      0.900     1.000     0.947       324
           1      0.000     0.000     0.000        36

    accuracy                          0.900       360
   macro avg      0.450     0.500     0.474       360
weighted avg      0.810     0.900     0.853       360

dummy confusion-matrix script parity: PASS


### 혼동행렬 해석 연습

위 결과를 보면:
- **TN=324**: 음성(0)을 음성으로 맞춤 → 많음 (좋음)
- **FP=0**: 음성을 양성으로 틀림 → 없음
- **FN=36**: 양성(1)을 음성으로 틀림 → **전부!** (문제!)
- **TP=0**: 양성을 양성으로 맞춤 → 없음 (큰 문제!)

> **양성을 하나도 못 찾았는데 정확도는 90%!**  
> 혼동행렬을 보면 문제가 바로 보임

---
## Part 4. 핵심 지표: Precision, Recall, F1

### 공식

| 지표 | 공식 | 의미 |
|------|------|------|
| **Accuracy** | (TP + TN) / 전체 | 전체 중 맞춘 비율 |
| **Precision** | TP / (TP + FP) | 양성 예측 중 실제 양성 |
| **Recall** | TP / (TP + FN) | 실제 양성 중 찾은 비율 |
| **F1** | 2 × (P × R) / (P + R) | Precision과 Recall의 조화평균 |

### 다시 정리 호호

- **Precision(정밀도)**: 양성이라고 했는데, 진짜 양성이 맞나?
    - 높으면 → **오탐(FP)이 적음**
    - 예: 스팸 필터가 정상 메일을 스팸으로 잘못 분류하면 안 됨

- **Recall(재현율)**: 진짜 양성 중에서 얼마나 찾았나?
    - 높으면 → **누락(FN)이 적음**
    - 예: 암 진단에서 암 환자를 놓치면 안 됨
- **F1**: Precision과 Recall의 조화평균
    - 둘 중 하나가 0이면 F1도 0에 가까워짐(균형을 강하게 요구)
    - 결국 P/R은 FP/FN 비용을 반영한 의사결정 지표임

![Recall: 놓치지 않았나?](스크린샷%202026-02-09%20오후%202.33.22.png)

![Precision: 헛발질 안 했나?](스크린샷%202026-02-09%20오후%202.33.36.png)

![F1 Score: 균형의 미학](스크린샷%202026-02-09%20오후%202.33.57.png)

### 업무에 따른 중요도

| 상황 | 더 중요한 지표                   | 이유 |
|------|----------------------------|------|
| **스팸 메일 필터** | Precision                  | 정상 메일을 스팸으로 분류하면 안 됨 |
| **암 진단** | Recall                     | 암 환자를 놓치면 생명 위험 |
| **금융 사기 탐지** | Recall                     | 사기를 놓치면 큰 손실 |
| **추천 시스템** | Precision/Recall/랭킹(TOP-K) | 잘못된 추천은 사용자 이탈 유발 |

> **무엇을 틀리면 더 큰 문제인가?**를 먼저 생각하셈

### 기준선 모델: Logistic Regression

여기서는 threshold tradeoff를 보여주기 위해 일부러 약하게 만든 모델이 아니라,
먼저 **일부러 약하게 만들지 않은 기준선**을 확인함.

알고리즘 내부 원리보다, 서로 다른 지표가 같은 예측을 어떻게 읽는지에 집중할 것.

In [8]:
from sklearn.linear_model import LogisticRegression

lr_7 = LogisticRegression(max_iter=5000, random_state=42).fit(X_train_7, y_train_7)
y_pred_lr_7 = lr_7.predict(X_test_7)
accuracy_lr_7 = accuracy_score(y_test_7, y_pred_lr_7)
print(f"Logistic Accuracy: {accuracy_lr_7:.4f}")

Logistic Accuracy: 0.9944


In [9]:
from sklearn.metrics import balanced_accuracy_score, precision_recall_fscore_support

balanced_acc_7 = balanced_accuracy_score(y_test_7, y_pred_lr_7)
prec_macro_7, rec_macro_7, f1_macro_7, _ = precision_recall_fscore_support(
    y_test_7, y_pred_lr_7, average="macro", zero_division=0
)
prec_weight_7, rec_weight_7, f1_weight_7, _ = precision_recall_fscore_support(
    y_test_7, y_pred_lr_7, average="weighted", zero_division=0
)

cm_lr_7 = confusion_matrix(y_test_7, y_pred_lr_7, labels=[0, 1])
precision_class7_7 = precision_score(y_test_7, y_pred_lr_7, zero_division=0)
recall_class7_7 = recall_score(y_test_7, y_pred_lr_7, zero_division=0)

print(f"Balanced Accuracy: {balanced_acc_7:.4f}")
print(f"Macro     P:{prec_macro_7:.4f} R:{rec_macro_7:.4f} F1:{f1_macro_7:.4f}")
print(f"Weighted  P:{prec_weight_7:.4f} R:{rec_weight_7:.4f} F1:{f1_weight_7:.4f}")
print("Baseline TN, FP, FN, TP:", *cm_lr_7.ravel())
print(f"Positive=7 Precision:{precision_class7_7:.4f} Recall:{recall_class7_7:.4f}")

expected_baseline_metric_parity_7 = [0.9944, 0.9722, 0.9842]
actual_baseline_metric_parity_7 = [
    round(accuracy_lr_7, 4),
    round(balanced_acc_7, 4),
    round(f1_macro_7, 4),
]
assert actual_baseline_metric_parity_7 == expected_baseline_metric_parity_7, actual_baseline_metric_parity_7
assert tuple(cm_lr_7.ravel()) == (324, 0, 2, 34), cm_lr_7.ravel()
assert round(precision_class7_7, 4) == 1.0000, precision_class7_7
assert round(recall_class7_7, 4) == 0.9444, recall_class7_7
assert round(rec_macro_7, 4) == round(balanced_acc_7, 4) == 0.9722
print("baseline metric and interpretation parity: PASS")

Balanced Accuracy: 0.9722
Macro     P:0.9969 R:0.9722 F1:0.9842
Weighted  P:0.9945 R:0.9944 F1:0.9944
Baseline TN, FP, FN, TP: 324 0 2 34
Positive=7 Precision:1.0000 Recall:0.9444
baseline metric and interpretation parity: PASS


### 같은 기준선, 서로 다른 Evidence

| 지표 | 실제 출력 |
|------|----------:|
| Accuracy | **0.9944** |
| Balanced Accuracy | **0.9722** |
| Macro F1 | **0.9842** |

먼저 현재 기준선의 혼동행렬을 **Positive=숫자 7** 기준으로 읽어봄.

```text
TN=324, FP=0, FN=2, TP=34
```

- Precision = 34 / (34 + 0) = **1.000**: `7`이라고 예측한 것 중 틀린 것은 없음.
- Recall = 34 / (34 + 2) ≈ **0.944**: 실제 `7` 가운데 두 개를 놓침.

그 다음 화면의 두 숫자를 비교함.

> **Balanced Accuracy 0.9722와 Macro Recall 0.9722가 왜 같을까?**

Balanced Accuracy는 **클래스별 Recall을 같은 비중으로 평균**한 값임.
Macro Recall도 클래스별 Recall을 같은 비중으로 평균하므로 두 지표는 **정의상 같은 값**임.
현재 화면의 `0.9722`는 그 정의가 이 digits 7-vs-rest 기준선 실험에서 실제로 확인된 관찰값임.

Macro Precision `0.9969`와 Macro Recall `0.9722`의 차이도 보이지만,
먼저 혼동행렬 → 숫자 7의 Precision/Recall → 클래스별 Recall 평균 순서로 읽을 것.

이건 기준선 모델이 나쁘다는 뜻이 아님.
**Accuracy 한 숫자만으로는 어떤 클래스를 놓쳤는지 읽을 수 없다는 뜻**임.

| 평균 방식 | 의미 | 언제 확인하나 |
|----------|------|---------------|
| **Balanced Accuracy** | 클래스별 Recall 평균 | 클래스별 놓침을 같은 비중으로 볼 때 |
| **Macro** | 클래스별 지표의 단순 평균 | 소수 클래스도 같은 비중으로 볼 때 |
| **Weighted** | 표본 수를 반영한 평균 | 실제 분포를 반영하되 다수 클래스 영향도 확인할 때 |

---
## Part 5. 임곗값(Threshold) = 정책 선택

분류 모델은 먼저 Positive일 **확률**을 출력하고,
그 확률을 어떤 기준에서 0과 1로 나눌지 정할 수 있음.

- 확률 ≥ threshold → Positive(1)
- 확률 < threshold → Negative(0)

threshold를 바꾸면 Positive로 예측하는 범위와 Precision / Recall / F1이 함께 달라짐.
어떤 tradeoff가 생기는지는 아래에서 먼저 예측한 뒤 Evidence로 확인할 것.

### train / validation / test 역할 회수

오늘 새로 만든 규칙이 아님. 지난 회차에서 나눈 역할을 threshold 선택에도 그대로 적용함.

> 배울 때는 **train**만 봄.  
> 선택할 때는 **validation**을 봄.  
> **test**는 모든 선택이 끝난 뒤 마지막에 한 번 봄.

Part 1~4에서는 이미 정해진 모델과 기준을 test에서 **평가하고 관찰**했을 뿐,
그 test 결과를 보고 모델·전처리·threshold를 다시 고르지 않았음. 앞부분을 잘못된 절차로 보는 것이 아님.

Part 5부터는 threshold를 **선택**해야 하므로 validation이 필요함.
선택에 사용한 데이터로 마지막 성능까지 평가하면, 더 이상 처음 보는 시험이라고 할 수 없음.
test 자체를 보는 것이 금지인 게 아니라, test 결과를 선택 과정의 피드백으로 사용하면
그 데이터는 **final test 역할을 잃는다는 것**이 핵심임.

Part 5는 앞의 기준선 확인과 **별도 실험**으로 시작하고, 새 test는 그대로 남겨둘 것.

In [10]:
# Part 5 전용 train / validation / test: data split seed 42
X_dev_threshold_7, X_final_test_7, y_dev_threshold_7, y_final_test_7 = train_test_split(
    X_all_7,
    y_all_7,
    test_size=0.2,
    stratify=y_all_7,
    random_state=42,
)
X_fit_7, X_val_7, y_fit_7, y_val_7 = train_test_split(
    X_dev_threshold_7,
    y_dev_threshold_7,
    test_size=0.25,
    stratify=y_dev_threshold_7,
    random_state=42,
)

print("train      :", X_fit_7.shape, "Positive=", int(y_fit_7.sum()))
print("validation :", X_val_7.shape, "Positive=", int(y_val_7.sum()))
print("test       :", X_final_test_7.shape, "Positive=", int(y_final_test_7.sum()), "(아직 선택에 사용하지 않음)")

train      : (1077, 64) Positive= 107
validation : (360, 64) Positive= 36
test       : (360, 64) Positive= 36 (아직 선택에 사용하지 않음)


### threshold tradeoff를 보기 위한 teaching setup

완벽한 모델에서는 threshold를 바꿔도 실수의 교환이 잘 보이지 않음.
그래서 이 파트에서는 **일부러 정보를 10개로 줄이고 규제를 준 모델**을 사용함.

`PCA(10)`과 `LogisticRegression(C=0.1)`의 내부 원리를 깊게 배우는 장면이 아님.
목적은 **같은 확률에서 threshold를 바꿀 때 어떤 실수가 늘고 줄어드는지** 읽는 것.

Pipeline 전체는 **train에만 fit**함. 따라서 안의 PCA도 test나 validation을 미리 보지 않음.

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

threshold_pipe_7 = Pipeline([
    ("pca", PCA(n_components=10, random_state=42)),
    ("logistic", LogisticRegression(max_iter=5000, C=0.1, random_state=42)),
])

threshold_pipe_7.fit(X_fit_7, y_fit_7)  # train에만 fit
proba_val_7 = threshold_pipe_7.predict_proba(X_val_7)[:, 1]

print("Pipeline fit 범위: train only")
print("validation probability 개수:", len(proba_val_7))

Pipeline fit 범위: train only
validation probability 개수: 360


### Selection Rule — 표를 보기 전에 정함

이번 실습에서는 학습을 단순하게 하기 위해
**validation F1이 가장 높은 threshold**를 선택하겠음.

실제 업무에서 F1 최대가 정답인 것은 아님.
놓치면 안 되는 문제라면 Recall 조건을 먼저 정할 수도 있고,
FP와 FN의 실제 비용을 기준으로 선택할 수도 있음.

### [3-2] Prediction Card 3

> **threshold를 낮추면 놓치는 Positive가 줄 때 대가는?**

예상한 뒤 validation 표를 확인할 것.

In [12]:
thresholds_7 = np.round(np.arange(0.1, 1.0, 0.1), 1)

validation_rows_7 = []
for threshold_7 in thresholds_7:
    y_pred_val_7 = (proba_val_7 >= threshold_7).astype(int)
    validation_rows_7.append({
        "threshold": threshold_7,
        "precision": precision_score(y_val_7, y_pred_val_7, zero_division=0),
        "recall": recall_score(y_val_7, y_pred_val_7, zero_division=0),
        "f1": f1_score(y_val_7, y_pred_val_7, zero_division=0),
        "accuracy": accuracy_score(y_val_7, y_pred_val_7),
    })

thr_table_7 = pd.DataFrame(validation_rows_7)
display(thr_table_7.round(3))

best_row_7 = thr_table_7.loc[thr_table_7["f1"].idxmax()]
selected_threshold_7 = float(best_row_7["threshold"])
print(
    f"[validation F1 최대] threshold={selected_threshold_7:.1f}, "
    f"P={best_row_7['precision']:.3f}, R={best_row_7['recall']:.3f}, "
    f"F1={best_row_7['f1']:.3f}"
)

,threshold,precision,recall,f1,accuracy
0,0.1,0.854,0.972,0.909,0.981
1,0.2,0.897,0.972,0.933,0.986
2,0.3,0.921,0.972,0.946,0.989
3,0.4,0.919,0.944,0.932,0.986
4,0.5,0.943,0.917,0.930,0.986
5,0.6,0.939,0.861,0.899,0.981
6,0.7,0.968,0.833,0.896,0.981
7,0.8,0.964,0.750,0.844,0.972
8,0.9,1.000,0.694,0.820,0.969


[validation F1 최대] threshold=0.3, P=0.921, R=0.972, F1=0.946


### 선택 완료 — test 최종 평가 1회

threshold는 validation 표에서 이미 확정했음.
이제 test를 처음 열어 **선택된 하나의 threshold만** 최종 평가함.
test 결과를 보고 threshold를 다시 바꾸지 않음.

In [13]:
proba_test_7 = threshold_pipe_7.predict_proba(X_final_test_7)[:, 1]
y_pred_test_threshold_7 = (proba_test_7 >= selected_threshold_7).astype(int)

test_precision_7 = precision_score(y_final_test_7, y_pred_test_threshold_7, zero_division=0)
test_recall_7 = recall_score(y_final_test_7, y_pred_test_threshold_7, zero_division=0)
test_f1_7 = f1_score(y_final_test_7, y_pred_test_threshold_7, zero_division=0)

print(f"선택된 threshold: {selected_threshold_7:.1f}")
print(f"FINAL TEST — Precision={test_precision_7:.3f}, Recall={test_recall_7:.3f}, F1={test_f1_7:.3f}")

선택된 threshold: 0.3
FINAL TEST — Precision=0.919, Recall=0.944, F1=0.932


### data split seed 반복 — 절차의 안정성 확인

이 표는 threshold를 **다시 고르기 위한 표가 아님.**
각 행은 threshold를 validation에서 결정한 **독립적인 반복 실험**임.

동일 절차를 여러 split에서 반복했을 때 선택과 결과가 어떻게 달라지는지 확인하는 분석이며,
**test 결과를 보고 threshold나 seed를 다시 선택하지 않음.**

이번 PCA + LogisticRegression 설정에서는 모델 random state가 아니라
**data split seed**를 바꾼 결과를 비교함.

In [14]:
def run_threshold_experiment_7(data_split_seed_7):
    X_dev_seed_7, X_test_seed_7, y_dev_seed_7, y_test_seed_7 = train_test_split(
        X_all_7,
        y_all_7,
        test_size=0.2,
        stratify=y_all_7,
        random_state=data_split_seed_7,
    )
    X_fit_seed_7, X_val_seed_7, y_fit_seed_7, y_val_seed_7 = train_test_split(
        X_dev_seed_7,
        y_dev_seed_7,
        test_size=0.25,
        stratify=y_dev_seed_7,
        random_state=data_split_seed_7,
    )

    pipe_seed_7 = Pipeline([
        ("pca", PCA(n_components=10, random_state=42)),
        ("logistic", LogisticRegression(max_iter=5000, C=0.1, random_state=42)),
    ])
    pipe_seed_7.fit(X_fit_seed_7, y_fit_seed_7)

    proba_seed_val_7 = pipe_seed_7.predict_proba(X_val_seed_7)[:, 1]
    f1_by_threshold_7 = [
        f1_score(y_val_seed_7, proba_seed_val_7 >= threshold_7, zero_division=0)
        for threshold_7 in thresholds_7
    ]
    selected_seed_threshold_7 = float(thresholds_7[int(np.argmax(f1_by_threshold_7))])

    # 각 독립 실험에서 validation 선택이 끝난 뒤 test를 정확히 한 번 평가
    proba_seed_test_7 = pipe_seed_7.predict_proba(X_test_seed_7)[:, 1]
    y_pred_seed_test_7 = (proba_seed_test_7 >= selected_seed_threshold_7).astype(int)

    return {
        "data split seed": data_split_seed_7,
        "validation에서 선택한 threshold": selected_seed_threshold_7,
        "test Precision": precision_score(y_test_seed_7, y_pred_seed_test_7, zero_division=0),
        "test Recall": recall_score(y_test_seed_7, y_pred_seed_test_7, zero_division=0),
        "test F1": f1_score(y_test_seed_7, y_pred_seed_test_7, zero_division=0),
    }

seed_rows_7 = [
    run_threshold_experiment_7(data_split_seed_7)
    for data_split_seed_7 in [1, 7]
]
# seed 42는 바로 위 walk-through에서 test를 이미 1회 평가했으므로 그 결과를 재사용
seed_rows_7.append({
    "data split seed": 42,
    "validation에서 선택한 threshold": selected_threshold_7,
    "test Precision": test_precision_7,
    "test Recall": test_recall_7,
    "test F1": test_f1_7,
})
seed_results_7 = pd.DataFrame(seed_rows_7)

expected_seed_parity_7 = [
    [1, 0.7, 1.000, 0.833, 0.909],
    [7, 0.4, 0.897, 0.972, 0.933],
    [42, 0.3, 0.919, 0.944, 0.932],
]
actual_seed_parity_7 = seed_results_7.round(3).values.tolist()
assert actual_seed_parity_7 == expected_seed_parity_7, actual_seed_parity_7

display(seed_results_7.round(3))
print("seed 1/7/42 parity: PASS")

,data split seed,validation에서 선택한 threshold,test Precision,test Recall,test F1
0,1,0.7,1.000,0.833,0.909
1,7,0.4,0.897,0.972,0.933
2,42,0.3,0.919,0.944,0.932


seed 1/7/42 parity: PASS


### Reveal

같은 F1 근처에서도 Precision과 Recall을 보면 모델의 행동은 꽤 다름.
threshold는 "점수를 높이는 숫자"가 아니라 **어떤 실수를 더 감수할지 정하는 기준**임.

고정된 정답 threshold가 있는 것도 아님. 데이터 split이 달라지면 선택도 흔들릴 수 있음.
중요한 건 **test를 보지 않고 선택했다는 절차**임.

In [15]:
fig_threshold_7 = go.Figure()
fig_threshold_7.add_trace(go.Scatter(
    x=thr_table_7["threshold"], y=thr_table_7["precision"],
    mode="lines", name="Precision", line=dict(dash="dash", color="cyan"),
))
fig_threshold_7.add_trace(go.Scatter(
    x=thr_table_7["threshold"], y=thr_table_7["recall"],
    mode="lines", name="Recall", line=dict(color="magenta"),
))
fig_threshold_7.add_trace(go.Scatter(
    x=thr_table_7["threshold"], y=thr_table_7["f1"],
    mode="lines", name="F1", line=dict(color="yellow"),
))
fig_threshold_7.add_vline(
    x=selected_threshold_7,
    line_dash="dot",
    line_color="white",
    annotation_text="validation 선택",
)
fig_threshold_7.update_layout(
    title="Validation: threshold에 따른 Precision / Recall / F1 (7 vs others)",
    xaxis_title="Threshold",
    yaxis_title="Score",
    yaxis=dict(range=[0, 1]),
    template="plotly_dark",
)
fig_threshold_7.show()

### [3-2] Prediction Card 3 Reveal + 선택 가이드

threshold를 낮추면 더 많은 샘플을 Positive로 잡으므로
놓치는 Positive(FN)는 줄어들 수 있지만, Negative를 Positive로 잘못 잡는 FP가 늘 수 있음.
그 대가가 validation 표에서 **Precision 변화**로 보임.

| 업무 기준 | 선택 방식의 예 |
|----------|----------------|
| 놓치지 않는 것이 우선 | Recall 요구조건을 먼저 정함 |
| 잘못 울리는 경보 비용이 큼 | FP 비용 또는 Precision을 함께 봄 |
| 두 실수를 비슷하게 다룸 | validation F1 최대를 후보로 볼 수 있음 |

> threshold에는 모두에게 같은 정답이 없음. **선택 규칙을 먼저 선언하고 validation에서 고름.**

---
## Part 6. ROC 곡선과 AUC, PR 곡선(AP) — validation에서 확인

### ROC 곡선이란?

- **X축**: FPR (False Positive Rate) = FP / (FP + TN)
- **Y축**: TPR (True Positive Rate) = Recall = TP / (TP + FN)
- 임곗값을 0에서 1까지 바꾸며 그린 곡선

**해석**:
- 곡선이 **왼쪽 위**로 갈수록 좋음
- **AUC** (Area Under Curve): 곡선 아래 면적
  - 0.5 = 랜덤
  - 1.0 = 완벽

> ROC-AUC는 **Positive와 Negative의 점수 분포가 통계적으로 얼마나 잘 구분되는지**를 보는 점수.
>
> threshold 하나를 고르기 전, **validation의 전체 기준 구간에서** 두 클래스를 얼마나 구분하는지 확인함.

![ROC의 핵심 질문: 기준을 바꿔도 흔들리지 않는가?](스크린샷%202026-02-09%20오후%202.35.45.png)

![ROC & AUC: 통계적 구분 능력](스크린샷%202026-02-13%20오후%202.17.10.png)

In [16]:
from sklearn.metrics import roc_curve, roc_auc_score
import numpy as np
import plotly.graph_objects as go

# threshold_pipe_7이 train에서 fit된 뒤 만든 validation 확률만 사용
proba_dummy_val_7 = np.zeros(len(y_val_7))

fpr_pipe_val_7, tpr_pipe_val_7, _ = roc_curve(y_val_7, proba_val_7)
auc_pipe_val_7 = roc_auc_score(y_val_7, proba_val_7)

fpr_dummy_val_7, tpr_dummy_val_7, _ = roc_curve(y_val_7, proba_dummy_val_7)
auc_dummy_val_7 = roc_auc_score(y_val_7, proba_dummy_val_7)

fig_roc_val_7 = go.Figure()
fig_roc_val_7.add_trace(go.Scatter(
    x=fpr_pipe_val_7, y=tpr_pipe_val_7,
    mode="lines", name=f"Clean Pipeline (AUC={auc_pipe_val_7:.3f})",
    line=dict(color="cyan", width=3),
))
fig_roc_val_7.add_trace(go.Scatter(
    x=fpr_dummy_val_7, y=tpr_dummy_val_7,
    mode="lines", name=f"Dummy (AUC={auc_dummy_val_7:.3f})",
    line=dict(color="red", dash="dash"),
))
fig_roc_val_7.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="Random (AUC=0.5)",
    line=dict(color="gray", dash="dot"),
))
fig_roc_val_7.update_layout(
    title="Validation ROC — clean Pipeline (digits: 7 vs others)",
    xaxis_title="FPR (False Positive Rate)",
    yaxis_title="TPR (Recall)",
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, 1]),
    template="plotly_dark",
)
fig_roc_val_7.show()
print(f"[validation ROC-AUC] clean Pipeline: {auc_pipe_val_7:.3f} | Dummy: {auc_dummy_val_7:.3f}")

[validation ROC-AUC] clean Pipeline: 0.997 | Dummy: 0.500


In [17]:
import pandas as pd
import plotly.express as px

auc_df_7 = pd.DataFrame({
    "Model": ["Dummy", "Clean Pipeline"],
    "Validation ROC-AUC": [auc_dummy_val_7, auc_pipe_val_7],
})

fig_auc_7 = px.bar(
    auc_df_7,
    x="Model",
    y="Validation ROC-AUC",
    text="Validation ROC-AUC",
    template="plotly_dark",
    range_y=[0, 1],
    title="Validation ROC-AUC 비교",
)
fig_auc_7.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig_auc_7.show()

![PR Curve & AP: 랭킹의 품격](스크린샷%202026-02-13%20오후%202.19.16.png)

### PR 곡선이란?

- **X축**: Recall
- **Y축**: Precision
- **불균형 데이터**에서 더 직관적

**해석**:
- 곡선이 **오른쪽 위**로 갈수록 좋음
- **AP** (Average Precision): PR 곡선 아래 면적
- 기준선 = 양성 비율 (예: 10%)

> **직관 한 줄**: AP는 **Positive를 점수 기준으로 정렬했을 때, 위쪽에 얼마나 모여 있는지** 보는 지표.
>
> 즉, **AP = Positive 랭킹 품질**!  
> Positive가 드문 상황에서도 "높은 점수를 받은 것들이 진짜 Positive인가?"를 평가함

In [18]:
from sklearn.metrics import precision_recall_curve, average_precision_score
import plotly.graph_objects as go

precision_curve_val_7, recall_curve_val_7, thresholds_curve_val_7 = precision_recall_curve(
    y_val_7, proba_val_7
)
ap_pipe_val_7 = average_precision_score(y_val_7, proba_val_7)
positive_rate_val_7 = y_val_7.mean()

fig_pr_val_7 = go.Figure()
fig_pr_val_7.add_trace(go.Scatter(
    x=recall_curve_val_7,
    y=precision_curve_val_7,
    mode="lines",
    name=f"Clean Pipeline PR (AP={ap_pipe_val_7:.3f})",
    line=dict(color="magenta", width=3),
))
fig_pr_val_7.add_trace(go.Scatter(
    x=[0, 1],
    y=[positive_rate_val_7, positive_rate_val_7],
    mode="lines",
    name=f"Validation positive rate={positive_rate_val_7:.2f}",
    line=dict(dash="dash", color="grey"),
))
fig_pr_val_7.update_layout(
    title="Validation Precision-Recall — clean Pipeline (7 vs others)",
    xaxis_title="Recall",
    yaxis_title="Precision",
    template="plotly_dark",
    yaxis=dict(range=[0, 1]),
    xaxis=dict(range=[0, 1]),
)
fig_pr_val_7.show()
print(f"[validation AP] clean Pipeline: {ap_pipe_val_7:.3f}")

[validation AP] clean Pipeline: 0.979


### ROC vs PR: 같은 validation Evidence를 두 관점으로 보기

| 곡선 | 중심 질문 | 이번 실습에서의 역할 |
|------|----------|----------------------|
| **ROC-AUC** | 두 클래스의 점수 순서가 얼마나 잘 갈리는가 | validation 전체 threshold 구간 확인 |
| **PR-AP** | Positive를 얼마나 잘 찾고, 그 예측이 얼마나 정확한가 | 불균형에서 Positive 중심 확인 |

> 둘 중 하나를 자동 정답으로 고르지 않음. **Positive 정의와 업무 비용에 맞춰 함께 해석함.**

In [19]:
import plotly.graph_objects as go

metrics = ["ROC Curve", "PR Curve"]
x_axis = [
    "FPR (False Positive Rate)",
    "Recall (재현율)"
]
y_axis = [
    "TPR (True Positive Rate, 재현율)",
    "Precision (정밀도)"
]
interpret = [
    "곡선이 왼쪽 위로 갈수록 좋음 (AUC↑)",
    "곡선이 오른쪽 위로 갈수록 좋음 (AP↑)"
]
when_use = [
    "클래스가 비교적 균형 있을 때 전체 성능 요약",
    "불균형 데이터에서 더 직관적"
]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["지표", "X축", "Y축", "해석", "언제 유용한가"],
        fill_color="purple", align="center", font=dict(color="white")
    ),
    cells=dict(
        values=[metrics, x_axis, y_axis, interpret, when_use],
        fill_color=[["black","black"]],
        align="center",
        font=dict(color="white")
    )
)])

fig.update_layout(
    title="ROC vs PR Curve 비교",
    template="plotly_dark"
)

fig.show()

---
## 오늘의 정리

| 개념 | 핵심 | 확인할 Evidence |
|------|------|-----------------|
| Accuracy | 전체 중 맞춘 비율 | 다른 지표와 함께 확인 |
| 혼동행렬 | TP/FP/FN/TN 분포 | 어떤 실수가 발생했는지 |
| Precision | Positive 예측 중 실제 Positive | FP 비용 |
| Recall | 실제 Positive 중 찾은 비율 | FN 비용 |
| F1 | Precision과 Recall의 균형 | 사전 선언한 선택 규칙 |
| threshold | 확률을 예측으로 바꾸는 기준 | validation에서 선택 |
| ROC-AUC / PR-AP | 여러 threshold에서의 구분과 Positive 중심 성능 | clean Pipeline의 validation 확률 |

### 평가 규율

> **평가를 위해 남겨둔 데이터의 정보를 학습 과정에서 미리 사용하지 않음.**

> 배울 때는 train만 봄.  
> 선택할 때는 validation을 봄.  
> test는 모든 선택이 끝난 뒤 마지막에 한 번 봄.

> **모델을 믿기 전에, 우리가 만든 실험을 먼저 의심해야 함.**

---
## 3회차 Ending

3-1에서는 **출력이 나왔다고 근거가 있는 것은 아니고, 점수가 나왔다고 평가를 믿을 수 있는 것도 아님**을 확인했음.

3-2에서는 **정확도가 높다고 우리가 필요한 일을 잘하는 것도 아님**을 확인했음.

> 숫자가 나왔다는 이유만으로 믿지 않음.  
> **무엇을 맞혔고, 무엇을 놓쳤고, 어떤 실수를 감수했는지** 확인함.

이 질문으로 3회차를 마침.

---

## 다음 시간 예고 — 4회차 | 분류 모델 열어보기

오늘까지는 **모델을 블랙박스로 두고** 나온 숫자를 어떻게 읽을지 배웠음.
`predict_proba`가 확률을 줬고, 우리는 그 확률에 임곗값을 걸었음.

**4회차에서는 그 확률이 어디서 나온 숫자인지 확인함.**

| | 무엇 | 기준 |
|---|---|---|
| **4-1** | 로지스틱 회귀 | **확률** — 점수 z → sigmoid |
| **4-2** | KNN | **거리** — 가까운 이웃 다수결 |

- **4-1**: 오늘 쓴 `predict_proba`의 내부를 열어봄. 계수(`coef_`)를 직접 꺼내서 "왜 이 확률이 나왔는지" 설명함
- **4-2**: 확률도 계수도 없는 모델을 만남. 그럼 무엇으로 판단하고, 무엇을 확인해야 하는지

그리고 오늘의 규율은 4회차에도 그대로 이어짐.

> 배울 때는 train만 봄.  
> 선택할 때는 validation을 봄.  
> test는 모든 선택이 끝난 뒤 마지막에 한 번 봄.

오늘은 **threshold**를 validation에서 골랐고,
4-1에서는 **threshold + 피처 + 인코딩**까지 validation에서 고름. **고를 것이 늘어남.**

---

> **오늘 Part 1의 Dummy(남자=사망, 정확도 77%)를 기억해두셈.**
> 4-1에서 만든 모델과 **다시 비교할 거임.**